<a href="https://colab.research.google.com/github/WoojinCho-Ryan/Studying-Pytorch/blob/main/herald_proofs_gpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets

In [ ]:
import torch
print(torch.cuda.is_available())  # True면 GPU 모드, False면 CPU 모드!


True


In [ ]:
from datasets import load_dataset
dataset = load_dataset('FrenzyMath/Herald_proofs')
print(dataset['train'][0])

{'id': 0, 'name': 'InitialSeg.eq', 'formal_theorem': 'theorem InitialSeg.eq [IsWellOrder β s] (f g : r ≼i s) (a) : f a = g a := by sorry', 'informal_theorem': 'Initial Segment Embeddings are Equal in Well-Ordered Types : For any types \\( \\alpha \\) and \\( \\beta \\) with relations \\( r \\) and \\( s \\) respectively, if \\( s \\) is a well-order on \\( \\beta \\), then for any two initial segment embeddings \\( f, g : r \\preceq_i s \\) and any element \\( a \\in \\alpha \\), it holds that \\( f(a) = g(a) \\).', 'formal_proof': 'theorem InitialSeg.eq [IsWellOrder β s] (f g : r ≼i s) (a) : f a = g a := by\n  rw [Subsingleton.elim f g]', 'informal_proof': 'Given that \\( s \\) is a well-order on \\( \\beta \\), the type of initial segment embeddings \\( r \\preceq_i s \\) is a subsingleton. This means that any two initial segment embeddings \\( f \\) and \\( g \\) from \\( r \\) to \\( s \\) are equal. Therefore, for any element \\( a \\in \\alpha \\), we have \\( f(a) = g(a) \\). Th

Goal: informal proof to formal proof

In [ ]:
print(dataset['train'][0]['informal_proof'])
print('-' * 100)
print(dataset['train'][0]['formal_proof'])

Given that \( s \) is a well-order on \( \beta \), the type of initial segment embeddings \( r \preceq_i s \) is a subsingleton. This means that any two initial segment embeddings \( f \) and \( g \) from \( r \) to \( s \) are equal. Therefore, for any element \( a \in \alpha \), we have \( f(a) = g(a) \). This is because the equality \( f = g \) implies \( f(a) = g(a) \) for all \( a \in \alpha \). Hence, the goal \( f(a) = g(a) \) is equivalent to \( g(a) = g(a) \), which is trivially true by the reflexive property of equality. This completes the proof.
----------------------------------------------------------------------------------------------------
theorem InitialSeg.eq [IsWellOrder β s] (f g : r ≼i s) (a) : f a = g a := by
  rw [Subsingleton.elim f g]


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token

def preprocess(ex):
  input_text = f"Prompt: {ex['informal_proof']}\nFormalization: "
  target_text = ex['formal_proof']
  enc = tokenizer(input_text + target_text, truncation=True, padding='max_length', max_length=512)
  return {**enc, 'labels': enc['input_ids']}

dataset_proc = dataset['train'].map(preprocess)

args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    logging_steps=10,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset_proc,
)
trainer.train()

Map:   0%|          | 0/44553 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: woojincho-cs (woojincho-ryan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.196500
20,2.461500
30,1.903200
40,1.633800
50,1.663000
60,1.603600
70,1.659700
80,1.539000
90,1.656300
100,1.430600


RuntimeError: [enforce fail at inline_container.cc:664] . unexpected pos 897873856 vs 897873744